In [0]:
# ============================================================
# BANKING DATA ENGINEERING PROJECT
# GOLD LAYER - ACCOUNT SUMMARY
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    max
)


# ============================================================
# SPARK SESSION
# ============================================================

spark = SparkSession.builder.appName(
    "Banking_Gold_Account_Summary"
).getOrCreate()


# ============================================================
# CONFIGURATION
# ============================================================

silver_path = (
    "/Volumes/bank_data/default/bank_data/"
    "silver_bank_transactions"
)

gold_path = (
    "/Volumes/bank_data/default/bank_data/"
    "gold_account_summary"
)


# ============================================================
# READ SILVER
# ============================================================

print("========================================")
print("         READ SILVER")
print("========================================")


silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)


silver_count = silver_df.count()


print(f"Silver Rows: {silver_count}")


# ============================================================
# GOLD TRANSFORMATION
# ============================================================

print("\n========================================")
print("      GOLD TRANSFORMATION")
print("========================================")


gold_df = (
    silver_df

    .groupBy("Account_No")

    .agg(
        count("*").alias("Transaction_Count"),

        sum(
            col("WITHDRAWAL_AMT")
        ).alias("Total_Withdrawal"),

        sum(
            col("DEPOSIT_AMT")
        ).alias("Total_Deposit"),

        avg(
            col("WITHDRAWAL_AMT")
        ).alias("Avg_Withdrawal"),

        avg(
            col("DEPOSIT_AMT")
        ).alias("Avg_Deposit"),

        max(
            col("DATE")
        ).alias("Last_Transaction_Date")
    )
)


# ============================================================
# WRITE GOLD
# ============================================================

print("\n========================================")
print("         WRITE GOLD")
print("========================================")


(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_path)
)


print(
    " Gold Account Summary written successfully"
)


# ============================================================
# READ GOLD FOR VALIDATION
# ============================================================

gold_check_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)


gold_count = gold_check_df.count()

gold_columns = len(
    gold_check_df.columns
)


# ============================================================
# GOLD VALIDATION
# ============================================================

print("\n========================================")
print("         GOLD VALIDATION")
print("========================================")


print(
    f"Silver Rows: {silver_count}"
)

print(
    f"Gold Account Rows: {gold_count}"
)

print(
    f"Gold Columns: {gold_columns}"
)


# ============================================================
# EXPECTED RESULTS
# ============================================================

expected_accounts = 10

expected_columns = 7


if gold_count == expected_accounts:

    print(
        " ACCOUNT COUNT VALIDATION PASSED"
    )

else:

    print(
        " ACCOUNT COUNT VALIDATION FAILED"
    )


if gold_columns == expected_columns:

    print(
        " COLUMN COUNT VALIDATION PASSED"
    )

else:

    print(
        " COLUMN COUNT VALIDATION FAILED"
    )


# ============================================================
# NULL VALIDATION
# ============================================================

print("\n========================================")
print("         NULL VALIDATION")
print("========================================")


null_accounts = (
    gold_check_df
    .filter(
        col("Account_No").isNull()
    )
    .count()
)


null_transaction_count = (
    gold_check_df
    .filter(
        col("Transaction_Count").isNull()
    )
    .count()
)


print(
    f"NULL Account_No: {null_accounts}"
)

print(
    f"NULL Transaction_Count: {null_transaction_count}"
)


if null_accounts == 0:

    print(
        " Account_No NULL CHECK PASSED"
    )

else:

    print(
        " Account_No NULL CHECK FAILED"
    )


if null_transaction_count == 0:

    print(
        " Transaction_Count NULL CHECK PASSED"
    )

else:

    print(
        "Transaction_Count NULL CHECK FAILED"
    )


# ============================================================
# ACCOUNT VALIDATION
# ============================================================

print("\n========================================")
print("      ACCOUNT SUMMARY VALIDATION")
print("========================================")


zero_transaction_accounts = (
    gold_check_df
    .filter(
        col("Transaction_Count") <= 0
    )
    .count()
)


print(
    f"Accounts with invalid transaction count: "
    f"{zero_transaction_accounts}"
)


if zero_transaction_accounts == 0:

    print(
        "TRANSACTION COUNT BUSINESS CHECK PASSED"
    )

else:

    print(
        " TRANSACTION COUNT BUSINESS CHECK FAILED"
    )


# ============================================================
# SCHEMA
# ============================================================

print("\n========================================")
print("         GOLD SCHEMA")
print("========================================")


gold_check_df.printSchema()


# ============================================================
# SAMPLE DATA
# ============================================================

print("\n========================================")
print("         GOLD SAMPLE DATA")
print("========================================")


display(
    gold_check_df.orderBy(
        col("Account_No")
    )
)


# ============================================================
# FINAL DATA QUALITY CHECK
# ============================================================

if (
    gold_count == expected_accounts
    and gold_columns == expected_columns
    and null_accounts == 0
    and null_transaction_count == 0
    and zero_transaction_accounts == 0
):

    print("\n========================================")
    print("   GOLD ACCOUNT SUMMARY COMPLETED")
    print("========================================")

    print(
        "ALL GOLD ACCOUNT CHECKS PASSED"
    )

    print(
        f"Accounts: {gold_count}"
    )

    print(
        f"Columns: {gold_columns}"
    )

    print(
        f"Gold Delta Path: {gold_path}"
    )

else:

    raise Exception(
        " GOLD ACCOUNT SUMMARY DATA QUALITY CHECK FAILED"
    )

         READ SILVER
Silver Rows: 116162

      GOLD TRANSFORMATION

         WRITE GOLD
 Gold Account Summary written successfully

         GOLD VALIDATION
Silver Rows: 116162
Gold Account Rows: 10
Gold Columns: 7
 ACCOUNT COUNT VALIDATION PASSED
 COLUMN COUNT VALIDATION PASSED

         NULL VALIDATION
NULL Account_No: 0
NULL Transaction_Count: 0
 Account_No NULL CHECK PASSED
 Transaction_Count NULL CHECK PASSED

      ACCOUNT SUMMARY VALIDATION
Accounts with invalid transaction count: 0
TRANSACTION COUNT BUSINESS CHECK PASSED

         GOLD SCHEMA
root
 |-- Account_No: string (nullable = true)
 |-- Transaction_Count: long (nullable = true)
 |-- Total_Withdrawal: double (nullable = true)
 |-- Total_Deposit: double (nullable = true)
 |-- Avg_Withdrawal: double (nullable = true)
 |-- Avg_Deposit: double (nullable = true)
 |-- Last_Transaction_Date: timestamp (nullable = true)


         GOLD SAMPLE DATA


Account_No,Transaction_Count,Total_Withdrawal,Total_Deposit,Avg_Withdrawal,Avg_Deposit,Last_Transaction_Date
1196428',48758,6.8449329070860054E10,6.830101070624993E10,4104907.2906063,2128884.789647163,2019-03-05T00:00:00.000Z
1196711',10518,4.6768840403000015E10,4.5838887934360054E10,9329511.351087177,8326773.466732072,2019-02-28T00:00:00.000Z
409000362497',29840,1.0193511991233997E11,1.0172093690494019E11,6586232.468329777,7082151.145647858,2019-03-05T00:00:00.000Z
409000405747',51,4.203178862E8,2.287852162E8,1.5011353078571428E7,9947183.313043477,2019-03-02T00:00:00.000Z
409000425051',802,3.845102355E8,2.703186353E7,1.3732508410714285E7,34924.88828165375,2019-03-02T00:00:00.000Z
409000438611',4588,4.705551279080001E9,4.705899408779997E9,3530045.9708027015,1445744.826046082,2019-03-05T00:00:00.000Z
409000438620',13454,1.719608322354001E10,1.7202434742950005E10,2049348.4952377558,3397676.2281157426,2019-03-05T00:00:00.000Z
409000493201',1044,9.537793092000014E7,9.565931423999996E7,165874.66246956546,203964.4226865671,2019-03-05T00:00:00.000Z
409000493210',6014,1.0060492432000004E8,1.0220961325000007E8,19247.163634972265,129872.44377382475,2019-03-05T00:00:00.000Z
409000611074',1093,1.45397419E8,1.45859619E8,186886.14267352186,463046.4095238095,2019-02-08T00:00:00.000Z



   GOLD ACCOUNT SUMMARY COMPLETED
ALL GOLD ACCOUNT CHECKS PASSED
Accounts: 10
Columns: 7
Gold Delta Path: /Volumes/bank_data/default/bank_data/gold_account_summary
